In [1]:
import re
import pandas as pd
import numpy as np
import sys

In [2]:
# Regular expression patterns for matching disorder transitions

pattern_disorder_disorder = r"\|derived-binding_mode_disorder_to_disorder-mobi"
pattern_disorder_order = r"\|derived-binding_mode_disorder_to_order-mobi"

# Function to extract transition scores from a mobidb fasta file

def get_transition_scores_from_mobidb(input_file, pattern_sequence=r"\|sequence\|"):
    '''
    Extracts sequences and their transition states from a mobidb fasta file.

    Args:
        input_file (str): Path to the input mobidb fasta file.
        pattern_sequence (str): Regex pattern to identify sequence headers.
    Returns a dictionary of {"sequence":"","transition_scores":""}
    disorder to disorder = 0, disorder to order = 1, anything else(no binding or not disodered) = 2
    '''
    import sys
    sys.set_int_max_str_digits(100000)  # Set the limit to 100000 digits
    sequences = []
    transition_scores = []
    disorder_disorder_score = None
    disorder_order_score = None
    id = None
    ids = []

    with open(input_file, "r") as infile:
        for line in infile:
            if re.search(pattern_sequence, line):
                # If not the first sequence, save previous scores
                if id is not None:
                    if disorder_disorder_score and disorder_order_score:
                        transition_score = str(int(disorder_order_score) + int(disorder_disorder_score))
                        transition_scores.append(transition_score.replace("2","0").replace("4", "2").replace("3", "1"))
                    elif disorder_disorder_score:
                        transition_scores.append(disorder_disorder_score)
                    elif disorder_order_score:
                        transition_scores.append(disorder_order_score)
                    else:
                        transition_scores.append("NaN")
                # Start new sequence
                id = line.strip().split("|")[0]
                ids.append(id.strip(">"))
                disorder_disorder_score = None
                disorder_order_score = None
                next_line = next(infile, None)
                if next_line:
                    sequences.append(next_line.strip())
                continue
            if id and re.search(re.escape(id) + pattern_disorder_disorder, line):
                next_line = next(infile, None)
                if next_line:
                    disorder_disorder_score = next_line.strip().replace("0", "2")
            if id and re.search(re.escape(id) + pattern_disorder_order, line):
                next_line = next(infile, None)
                if next_line:
                    disorder_order_score = next_line.strip().replace("0", "2").replace("1", "0")
        # After last sequence
        if id is not None:
            if disorder_disorder_score and disorder_order_score:
                transition_score = str(int(disorder_order_score) + int(disorder_disorder_score))
                transition_scores.append(transition_score.replace("2","0").replace("4", "2").replace("3", "1"))
            elif disorder_disorder_score:
                transition_scores.append(disorder_disorder_score)
            elif disorder_order_score:
                transition_scores.append(disorder_order_score)
            else:
                transition_scores.append("NaN")

    return {"Id":ids,"sequences": sequences, "transition_scores": transition_scores}

In [3]:
# Read a FASTA file and return a dictionary of sequences and ids.

def read_fasta(file_path):
    sequences=[]
    current_sequence=[]
    ids = []
    with open(file_path,"r") as file:
        for line in file:
            if line.startswith(">"):
                ids.append(line.strip().strip(">"))
                if current_sequence:
                    sequences.append(''.join(current_sequence))
                    current_sequence=[]
            else:
                current_sequence.append(line.strip())
        if current_sequence:
            sequences.append(''.join(current_sequence))
    return {"sequences": sequences, "Id": ids}

In [ ]:
# Loading data from mobidb fasta files and creating a DataFrame
data = get_transition_scores_from_mobidb("mobidb_search_2025-07-09T19-37-03_DD.fasta")
df_1 = pd.DataFrame(data)
data = get_transition_scores_from_mobidb("mobidb_search_2025-07-09T19-39-02_DO.fasta")
df_2 = pd.DataFrame(data)
df = pd.concat([df_1, df_2], ignore_index=True)
# Dropping duplicates based on Id, keeping the first occurrence
df = df.drop_duplicates(subset=["Id"], keep="first")
df.head(5)

# Saving the combined DataFrame to a CSV file
df.to_csv("all_mobidb_scores.csv", index=False)

In [ ]:
# Filter out sequences longer than 1000 characters

df["length"] = df["sequences"].apply(len)

print("Number of removed sequences:" ,df[df["length"] > 1000]["sequences"].count())
df_filtered = df[df["length"] <= 1000].reset_index(drop=True)

# Further filtering based on transition scores

def percent_score_num(scores,num):
    scores_arr = np.array(list(scores), dtype=int)
    return np.mean(scores_arr == num)

df_filtered = df_filtered[df_filtered["transition_scores"].apply(lambda scores: percent_score_num(scores, 2)) <= 0.7].reset_index(drop=True)

print(f"Remaining sequences: {len(df_filtered)}")

# Writing the processed data to a new fasta file
def write_fasta(df, output_file):
    with open(output_file, "w") as f:
        for index, row in df.iterrows():
            f.write(f">{row['Id']}\n{row['sequences']}\n")
write_fasta(df_filtered, "output_filtered_mobidb_scores_new.fasta")

# Reducing redundancy using cd-hit (uncomment and run in a suitable environment)
# !apt install cd-hit
#!cd-hit -i output_filtered_mobidb_scores_new.fasta -o output_mobidb_scores_cdhit_new.fasta -c 0.7

# Reading clustered data and saving to a DataFrame

In [ ]:
# Reading filtered and clustered data and saving to a DataFrame
clustered_mobidb_data = read_fasta("output_mobidb_scores_cdhit_new.fasta")
clustered_mobidb_df = pd.DataFrame(clustered_mobidb_data)
clustered_mobidb_df.head(5)

,sequences,Id
0,MFSFIDDIPSFEQIKARVRDDLRKHGWEKRWNDSRLVQKSRELLND...,P16536
1,MPYKLKKEKEPPKVAKCTAKPSSSGKDGGGENTEEAQPQPQPQPQP...,Q14738
2,CLGVGSCNDFAGCGYAIVCFW,P85078
3,MVTPALQMKKPKQFCRRMGQKKQRPARAGQPHSSSDAAQAPAEQPH...,P29372
4,MSGGSSCSQTPSRAIPATRRVVLGDGVQLPPGDYSTTPGGTLFSTT...,Q13541


In [ ]:
# Loading the original full dataset to merge transition scores
df_all = pd.read_csv("all_mobidb_scores.csv")

# Merging clustered data with original transition scores
df_clustered = clustered_mobidb_df.merge(df_all[["Id", "transition_scores"]], on="Id", how="left")
print(f"Total clustered sequences: {len(df_clustered)}")
# Saving the final processed DataFrame to a CSV file
df_clustered.to_csv("processed_mobidb_data.csv", index=False)

Total clustered sequences: 2886
